In [ ]:
import illustris_python as il
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib import cm
from matplotlib.cm import ScalarMappable
import seaborn as sns
import random
import pickle
from collections.abc import Iterable
from tqdm.notebook import tqdm
import pickle
import pandas as pd

from scipy.signal import find_peaks, medfilt, savgol_filter
from scipy.optimize import root_scalar, curve_fit
from scipy.optimize import brentq
from scipy.interpolate import UnivariateSpline, interp1d
from scipy.integrate import quad

from colossus.cosmology import cosmology
from colossus.halo import profile_nfw
from astropy.cosmology import Planck15
import astropy.units as u

# path to access TNG-50-1-Dark output data when on TNG Jupyter lab
basePath = '/home/tnguser/sims.TNG/TNG50-1-Dark/output/'

In [ ]:
# important parameters
# dimensionless hubble parameter
h = 0.6774
# Newton's gravitational constant
G = 4.30091e-6  # kpc (km/s)^2 / Msun
# Mass of dark matter particles in TNG-50-1-Dark
p_mass = 3.64755609660833*10**5 / h #Msun

### Gather host and subhalo data with preliminary filters on hosts

In [ ]:
# Load Group Catalog with structural parameters for relaxation tracking
group_fields = ['GroupFirstSub', 'GroupMass', 'GroupNsubs', 'GroupCM', 'GroupPos', 'Group_R_Crit200']
group_data = il.groupcat.loadHalos(basePath, 99, fields=group_fields)

# Load Subhalo masses at snap 99 to calculate substructure mass fraction (f_sub)
sub_masses = il.groupcat.loadSubhalos(basePath, 99, fields=['SubhaloMass'])

# Lists to hold the IDs of hosts that pass physical criteria
valid_hosts = []

# minimum host mass to ensure resolution of host and no dynamical friction effects
# 3000 is the recommended particle minimum
# 100 is the ratio of host to sub mass for negligible dynamical friction
min_host_mass = p_mass * 100 * 3000

print("Filtering hosts for equilibrium and relaxation criteria...")
# loop over hosts
for i in tqdm(range(len(group_data['GroupMass']))):
    # 1. Physical Mass Condition (converted to Msol)
    m_fof_msol = group_data['GroupMass'][i] * 1e10 / h
    if m_fof_msol < min_host_mass:
        continue
        
    # Skip if group has no subhalos assigned
    first_sub_idx = group_data['GroupFirstSub'][i]
    if first_sub_idx == -1:
        continue

    # 2. Center of Mass Offset (X_off < 0.07)
    pos = group_data['GroupPos'][i]
    cm = group_data['GroupCM'][i]
    r_200 = group_data['Group_R_Crit200'][i]
    
    if r_200 == 0: continue
    x_off = np.linalg.norm(cm - pos) / r_200
    if x_off > 0.07:
        continue  # Unrelaxed: skip
        
    # 3. Substructure Mass Fraction (f_sub < 0.1)
    m_primary = sub_masses[first_sub_idx]
    f_sub = (group_data['GroupMass'][i] - m_primary) / group_data['GroupMass'][i]
    if f_sub > 0.1:
        continue  # Has an overly dominant secondary/unrelaxed: skip
        
    # If it passes all checks, save the host dictionary
    valid_hosts.append({
        'GroupID': i,
        'GroupMass_Msol': m_fof_msol,
        'GroupFirstSub': first_sub_idx,
        'GroupNsubs': group_data['GroupNsubs'][i]
    })

print(f"Found {len(valid_hosts)} relaxed hosts above the mass threshold.")

In [ ]:
# Initialize bins for sampling hosts
bins = {
    '10^11': [], # < 10^12 Msol
    '10^12': [], # 10^12 to 10^13 Msol
    '10^13': [], # 10^13 to 10^14 Msol
    '10^14': []  # > 10^14 Msol
}

# Sort valid hosts into their respective bins
for host in valid_hosts:
    m = host['GroupMass_Msol']
    if m < 1e12:
        bins['10^11'].append(host)
    elif 1e12 <= m < 1e13:
        bins['10^12'].append(host)
    elif 1e13 <= m < 1e14:
        bins['10^13'].append(host)
    elif m >= 1e14:
        bins['10^14'].append(host) # there are no hosts here in TNG-50-1-Dark

# Define sample quotas per bin
quotas = {
    '10^11': 15,
    '10^12': 15,
    '10^13': 4,
}

selected_hosts = []
random.seed(42) # For reproducibility

# select hosts for each bin
for bin_name, quota in quotas.items():
    available = bins[bin_name]
    if len(available) <= quota:
        selected_hosts.extend(available)
        print(f"Bin {bin_name}: Taking all {len(available)} available hosts.")
    else:
        selected_hosts.extend(random.sample(available, quota))
        print(f"Bin {bin_name}: Randomly sampled {quota} out of {len(available)} hosts.")

print(f"Total master host sample size: {len(selected_hosts)} hosts.")

In [ ]:
# Global snapshot-to-scale-factor map to avoid redundant server lookups
# This creates a quick look-up dictionary for unit conversions
cosmo = cosmology.setCosmology("planck15")

snap_to_a = {}
snap_to_red = {}
snap_to_time = {}
for snap in tqdm(range(100)):
    header = il.groupcat.loadHeader(basePath, snap)
    redshift = header['Redshift']
    snap_to_red[snap] = redshift
    snap_to_a[snap] = 1 / (1 + redshift)
    snap_to_time[snap] = cosmo.age(redshift)

In [ ]:
# Optionally download these conversions for external use
snap_conversions = {}
snap_conversions['a'] = snap_to_a
snap_conversions['red'] = snap_to_red
snap_conversions['time'] = snap_to_time
with open('snap_conversions.pkl', 'wb') as f:
    pickle.dump(snap_conversions,f,protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# useful functions for unit conversions and other utilities

# convert snapshot number to id in snapshot array for that tree
def snap_to_id(snap_num, tree):
    return np.where(tree['SnapNum'] == snap_num)[0][0]

# convert comoving kpc / h to physical kpc
def to_physical(raw_dist, snap):
    """Converts comoving ckpc/h to physical kpc using the explicit snapshot scale factor."""
    a = snap_to_a[snap]
    return np.array(raw_dist) * a / h

# get infall snapshot of subhalo by the snapshot of peak mass
# this is more stable than defining it at a time after mass loss begins
def inf_mpeak_dynamic(sub_trees):
    inf_snap, rmax_inf, vmax_inf, mass_inf, vdisp_inf, vel_inf, pos_inf = [], [], [], [], [], [], []
    
    for tree in sub_trees:
        mpeak_ind = np.argmax(tree['SubhaloMass'])
        snap = tree['SnapNum'][mpeak_ind]
        
        inf_snap.append(snap)
        rmax_inf.append(to_physical(tree['SubhaloVmaxRad'][mpeak_ind], snap))
        vmax_inf.append(tree['SubhaloVmax'][mpeak_ind])
        mass_inf.append(tree['SubhaloMass'][mpeak_ind]*1e10/h)
        vdisp_inf.append(tree['SubhaloVelDisp'][mpeak_ind])
        vel_inf.append(tree['SubhaloVel'][mpeak_ind])
        pos_inf.append(to_physical(tree['SubhaloPos'][mpeak_ind], snap))
        
    return inf_snap, rmax_inf, vmax_inf, mass_inf, vdisp_inf, vel_inf, pos_inf

In [ ]:
# Load in subhalo data for each host and store

# Create a directory for checkpoints if it doesn't already exist
checkpoint_dir = "tng_raw_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

tree_fields = ['SubhaloMass', 'SubhaloID', 'SubfindID', 'SnapNum', 'SubhaloMassType', 
               'SubhaloHalfmassRad', 'SubhaloVmax', 'SubhaloVmaxRad', 'SubhaloGrNr', 
               'SubhaloPos', 'SubhaloVelDisp', 'SubhaloVel']

project_data_store = {}
random.seed(42)

print("Gathering host and satellite trees (with local checkpoint verification)...")

for host in tqdm(selected_hosts, desc="Processing Host Trees"):
    host_id = host['GroupID']
    checkpoint_filename = f"{checkpoint_dir}/host_{host_id}_raw.pkl"
    
    # --- CHECKPOINT FEATURE: Check if this host was already downloaded ---
    if os.path.exists(checkpoint_filename):
        with open(checkpoint_filename, 'rb') as f:
            project_data_store[host_id] = pickle.load(f)
        continue  # Skip server download and move to the next host
        
    # --- SERVER DOWNLOAD BLOCK ---
    # 1. Fetch Central Subhalo Tree
    cent_tree = il.sublink.loadTree(basePath, 99, host['GroupFirstSub'], fields=tree_fields, onlyMPB=True)
    if cent_tree is None: 
        continue
        
    # 3. Fetch Satellite Trees
    sub_ids = list(range(host['GroupFirstSub'] + 1, host['GroupFirstSub'] + host['GroupNsubs']))
        
    host_sub_trees = []
    for s_id in sub_ids:
        t = il.sublink.loadTree(basePath, 99, s_id, fields=tree_fields, onlyMPB=True)
        if t is not None:
            host_sub_trees.append(t)
            
    if len(host_sub_trees) == 0:
        continue 
        
    # 4. Process Infall Metrics
    inf_snap, rmax_inf, vmax_inf, mass_inf, vdisp_inf, vel_inf, pos_inf = inf_mpeak_dynamic(host_sub_trees)
    
    # Pack data for this specific host
    host_data = {
        'host_metadata': host,
        'central_tree': cent_tree,
        'satellite_trees': host_sub_trees,
        'infall_metrics': {
            'snap': inf_snap, 'rmax': rmax_inf, 'vmax': vmax_inf, 
            'mass': mass_inf, 'vdisp': vdisp_inf, 'vel': vel_inf, 'pos': pos_inf
        }
    }
    
    # --- CHECKPOINT FEATURE: Save this host's data immediately ---
    with open(checkpoint_filename, 'wb') as f:
        pickle.dump(host_data, f, protocol=pickle.HIGHEST_PROTOCOL)
        
    # Add to our active session store
    project_data_store[host_id] = host_data

print(f"\nAll downloads completed. Raw trees cached in '{checkpoint_dir}/'.")

### Filter remaining hosts and subhalos for resolved subhalos undergoing tidal evolution in a stable host

In [ ]:
# Final storage for pristine hosts and their highly qualified subhalo tracks
final_analysis_store = {}

print("Beginning multi-host and subhalo structural screening...")

# Loop over hosts and apply criteria
for host_id, data in tqdm(project_data_store.items(), desc="Filtering Systems"):
    metadata = data['host_metadata']
    cent_tree = data['central_tree']
    sat_trees = data['satellite_trees']
    infall = data['infall_metrics']
    
    # ----------------------------------------------------
    # CRITERION 1: HOST-LEVEL MASSIVE SATELLITE CUT (z=0)
    # ----------------------------------------------------
    # Check if a massive satellite breaks host potential symmetry
    if metadata['GroupNsubs'] > 1:
        largest_satellite_idx = metadata['GroupFirstSub'] + 1
        # Fetch its mass at snap 99 directly from the subhalo catalog data
        m_sat_z0 = sub_masses[largest_satellite_idx] # already in catalog units
        m_host_z0 = metadata['GroupMass_Msol'] / (1e10 / h) # convert back to catalog units
        
        if (m_sat_z0 / m_host_z0) > 0.1:
            continue # Skip host: Too asymmetric due to a massive satellite
            
    # ----------------------------------------------------
    # CRITERION 2: HOST-LEVEL TEMPORAL STABILITY
    # ----------------------------------------------------
    host_snaps = cent_tree['SnapNum']
    host_masses = cent_tree['SubhaloMass']
    snap_safe = 40 # Lower bound threshold, snapshots before are prone to unformed substructure
    
    # loop over snapshots, check for significant mass disturbance (e.g. major merger)
    for idx in range(len(host_snaps) - 1):
        snap_current = host_snaps[idx]
        if snap_current < 40:
            break
        m_later = host_masses[idx]
        m_earlier = host_masses[idx + 1]
        
        if m_earlier > 0:
            delta_m = (m_later - m_earlier) / m_earlier
            if delta_m > 0.15: # Major merger detected
                snap_safe = snap_current + 5 # Require a stabilization window
                break

    # ----------------------------------------------------
    # PRE-COMPUTE HOST HISTORICAL TRACKS FOR DISTANCE CALCS
    # ----------------------------------------------------
    # Build dictionary map for this specific host's positions across time using the central tree
    host_pos_map = {}
    for snap, pos in zip(cent_tree['SnapNum'], cent_tree['SubhaloPos']):
        host_pos_map[snap] = to_physical(pos, snap)

    # Get z=0 critical radius
    raw_r_crit = group_data['Group_R_Crit200'][host_id]
    r_crit_z0 = to_physical(raw_r_crit, 99)
    
    # Check for invalid critical radius
    if r_crit_z0 == 0:
        continue

    # Initialize storage for subhalos belonging to this host that clear all cuts
    qualified_sat_trees = []
    qualified_infall_metrics = {k: [] for k in infall.keys()}
    
    # ----------------------------------------------------
    # LOOP THROUGH SATELLITE SUBHALOS FOR STRUCTURAL CUTS
    # ----------------------------------------------------
    for sub_idx, s_tree in enumerate(sat_trees):
        sub_inf_snap = infall['snap'][sub_idx]
        
        # Check basic environment timeline and survival requirements
        if sub_inf_snap <= snap_safe or 99 not in s_tree['SnapNum']:
            continue
            
        # CUT A: 300 Particle Survival Cut at z=0 (Index 0 in tree data array)
        # SubhaloMass is in units of 10^10 Msol/h
        n_particles_z0 = (s_tree['SubhaloMass'][0] * 1e10 / h) / p_mass
        if n_particles_z0 < 300:
            continue
            
        # Calculate chronological physical distance profile from infall snapshot to snap 99
        sub_distances = []
        snaps_tracked = range(sub_inf_snap, 100)
        valid_profile = True
        
        for snap in snaps_tracked:
            if snap in s_tree['SnapNum'] and snap in host_pos_map:
                # convert snapshot to ID (in case of missing snapshots that offset indices)
                t_ind = np.where(s_tree['SnapNum'] == snap)[0][0]
                # get host and subhalo positions
                sub_pos_phys = to_physical(s_tree['SubhaloPos'][t_ind], snap)
                h_pos_phys = host_pos_map[snap]
                #calculate distance
                distance = np.linalg.norm(sub_pos_phys - h_pos_phys)
                sub_distances.append(distance)
            else:
                # If tree profile breaks chronological continuity, flag it
                valid_profile = False
                break
                
        # apply criteria and ensure sufficient data for the subhalo
        if not valid_profile or len(sub_distances) < 3:
            continue
            
        # CUT B: Containment Boundary Cut at z=0, distance must be less than r_crit (Snapshot 99 is the final index)
        if sub_distances[-1] > r_crit_z0:
            continue
            
        # CUT C: Complete Pericenter Orbit Filter (Peak -> Valley -> Peak)
        peaks, _ = find_peaks(sub_distances)
        valleys, _ = find_peaks(-np.array(sub_distances))
        
        has_completed_orbit = False
        for i in range(1, len(peaks)):
            left_peak = peaks[i - 1]
            right_peak = peaks[i]
            for valley in valleys:
                if left_peak < valley < right_peak:
                    has_completed_orbit = True
                    break
            if has_completed_orbit:
                break
                
        # If the subhalo survives every gatekeeper, store its track metrics
        if has_completed_orbit:
            qualified_sat_trees.append(s_tree)
            for key in infall.keys():
                qualified_infall_metrics[key].append(infall[key][sub_idx])

    # If the host retains valid tracks after strict pruning, add it to our final study
    if len(qualified_sat_trees) > 0:
        final_analysis_store[host_id] = {
            'host_metadata': metadata,
            'host_rcrit_z0': r_crit_z0,
            'satellite_trees': qualified_sat_trees,
            'central_tree': cent_tree,
            'infall_metrics': qualified_infall_metrics
        }

print(f"\nFiltering complete! Retained {len(final_analysis_store)} hosts with verified subhalo tidal tracks.")

In [ ]:
# save valid subhalo data
with open("valid_subhalos_multihost.pkl", "wb") as f:
    pickle.dump(final_analysis_store, f, protocol=pickle.HIGHEST_PROTOCOL)

### Calculate and Save Structural Parameters Not Included in Catalogue

In [ ]:
def get_vr_vt(sub_pos, sub_vel, host_pos, host_vel):
    """
    Calculates radial and tangential velocity components of the subhalo relative to the host.
    Expects physical positions and velocities.
    sub_pos: subhalo position
    sub_vel: subhalo velocity
    host_pos: host position
    host_vel: host velocity
    """
    # Relative position and velocity
    pos = sub_pos - host_pos
    vel = sub_vel - host_vel

    r_norm = np.linalg.norm(pos)
    
    # Handle the rare case where subhalo sits precisely at the host center
    if r_norm == 0:
        return np.nan, np.linalg.norm(vel)

    # Unit vector for radial direction
    r_hat = pos / r_norm
    
    # Projections
    v_r = np.dot(vel, r_hat)             # scalar radial velocity
    v_r_vec = v_r * r_hat                # radial component vector
    v_t_vec = vel - v_r_vec              # tangential component vector
    v_t = np.linalg.norm(v_t_vec)        # tangential speed
        
    return v_r, v_t

In [ ]:
#Finding subhalo concentrations
def nfw_mass(c, rhos, rs):
    """NFW mass within virial radius as a function of concentration c."""
    return 4 * np.pi * rhos * rs**3 * (np.log(1 + c) - c / (1 + c))

def solve_concentration(M_sub, rhos, rs, c_guess=10):
    """Solve for concentration c given subhalo mass, rhos, and rs."""
    def f(c):
        return nfw_mass(c, rhos, rs) - M_sub

    # Use root_scalar to find c in a reasonable range
    try:
        sol = root_scalar(f, bracket=[1, 100], method='brentq')
        if sol.converged:
            return sol.root
        else:
            return np.nan
    except ValueError:
        print("Invalid Concentration Encountered")
        return np.nan

# main function to get concentration
def get_concentration(tree, snap):
    """
    tree: the subhalo tree for which to calculate the concentration
    snap: snapshot to calculate concentration at
    """
    G = 4.301e-6  # (km/s)^2 kpc / M_solar
    snap_id = snap_to_id(snap,tree)
    M_sub = tree['SubhaloMass'][snap_id]*1e10 / h #convert to Msol
    rmax = to_physical(tree['SubhaloVmaxRad'][snap_id],snap)
    vmax = tree['SubhaloVmax'][snap_id]
    rs = rmax / 2.16
    rhos = 1/G *(vmax / (1.64 * rs))**2
    
    if rmax <= 0 or vmax <= 0:
        return np.nan
    
    return solve_concentration (M_sub,rhos,rs)

In [ ]:
#calculate velocity anisotropy constant
def get_anisotropy(tree,snap):
    """
    tree: the subhalo tree for which to calculate the concentration
    snap: snapshot to calculate concentration at
    """
    snap_id = snap_to_id(snap,tree)
    
    a = snap_to_a[snap]
    
    #get subhalo particle data
    dark = il.snapshot.loadSubhalo(basePath,snap,tree['SubfindID'][snap_id],1)
    if dark is None or 'Coordinates' not in dark:
        return np.nan # Safety catch
    pos = to_physical(dark['Coordinates'],snap) # kpc, stored ckpc/h
    vel = dark['Velocities'] * np.sqrt(a) # km / s (stored km root(a) / s)
    
    #subhalo center (potential minimum) position and bulk velocity
    cent_pos = to_physical(tree['SubhaloPos'][snap_id],snap)
    cent_vel = tree['SubhaloVel'][snap_id]

    # Test case, should output ~0
    # N = 100000
    # vel = np.random.normal(0, 1, (N, 3))
    # pos = np.random.normal(0, 1, (N, 3))
    # cent_pos = np.zeros(3)
    # cent_vel = np.zeros(3) 
    
    r_vec = pos - cent_pos        # shape (3,)
    r = np.linalg.norm(r_vec,axis=1)
    
    v_rel = vel - cent_vel        # shape (3,)
    
    #mask out the central particle (r=0)
    mask = r > 0
    x, y, z = r_vec[mask].T
    vx, vy, vz = v_rel[mask].T
    r = r[mask]

    # radial component
    v_r = (x*vx + y*vy + z*vz)/r  

    # tangential components
    v_theta = ((z*(x*vx + y*vy) - (x**2 + y**2)*vz) / (r*np.sqrt(x**2 + y**2)))
    v_phi = (-y*vx + x*vy) / np.sqrt(x**2 + y**2)
    
    sigma_r2 = np.var(v_r)
    sigma_theta2 = np.var(v_theta)
    sigma_phi2 = np.var(v_phi)

    beta = 1 - (sigma_theta2 + sigma_phi2) / (2 * sigma_r2)
    
    return beta

In [ ]:
#sphericity computation
def compute_shape(tree,snap_num):
    """
    tree: the subhalo tree for which to calculate the concentration
    snap_num: snapshot to calculate concentration at
    """
    # --- Step 1: Get subhalo position ---
    #get subhalo subfind id
    snap_id = snap_to_id(snap_num,tree)
    subhalo_id = tree['SubfindID'][snap_id]
    subhalo_pos = to_physical(tree['SubhaloPos'][snap_id],snap_num) #kpc

    # --- Step 2: Load DM particle coordinates for the subhalo ---
    subhalo_dm_data = il.snapshot.loadSubhalo(basePath, snap_num, subhalo_id, 1)
    dm_positions = to_physical(subhalo_dm_data['Coordinates'],snap_num)  # shape: (N_particles, 3), kpc

    # --- Step 3: Compute relative positions (centered on subhalo center) ---
    relative_positions = dm_positions - subhalo_pos
    
    # --- Step 4: Compute and display shape info ---
    shape = compute_axis_ratios(relative_positions)
    return shape
    
def compute_shape_tensor(positions, normalize=True):
    """
    positions: the positions of the subhalo's particles
    normalize: True means to use a normalized shape tensor, false means to use the absolute tensor
    """
    x, y, z = positions.T
    r2 = x**2 + y**2 + z**2
    if normalize:
        w = 1.0 / np.maximum(r2, 1e-8)
    else:
        w = np.ones_like(r2)
    S_xx = np.sum(w * x * x)
    S_yy = np.sum(w * y * y)
    S_zz = np.sum(w * z * z)
    S_xy = np.sum(w * x * y)
    S_xz = np.sum(w * x * z)
    S_yz = np.sum(w * y * z)
    return np.array([
        [S_xx, S_xy, S_xz],
        [S_xy, S_yy, S_yz],
        [S_xz, S_yz, S_zz]
    ])

# compute axis ratios (sphericity and triaxiality) using the shape tensor
def compute_axis_ratios(positions):
    """
    positions: subhalo particle positions
    """
    T = compute_shape_tensor(positions)
    eigvals = np.linalg.eigvalsh(T)  # always returns real sorted eigenvalues
    eigvals = np.clip(eigvals, 0, None)  # prevent negatives due to rounding
    a, b, c = np.sqrt(eigvals[::-1])     # sorted descending

    sphericity = c / a
    triaxiality = (a**2 - b**2) / (a**2 - c**2 + 1e-8)  # avoid divide-by-zero
    return {
        'a': a, 'b': b, 'c': c,
        'sphericity': sphericity,
        'triaxiality': triaxiality
    }

In [ ]:
# calculate the rotation curve for the subhalo and perform a univariate spline smoothing
def rot_curve(tree,snap,particle_data,N_bins=50,k=4):
    """
    tree: the subhalo tree for which to calculate the concentration
    snap: snapshot to calculate concentration at
    particle_data: output of the groupcat loadSubhalo function (dark matter only)
    N_bins: number of bins to use in binning the rotation curve
    k: degree of smoothing for the spline
    """
    snap_id = snap_to_id(snap,tree)
    coords = to_physical(particle_data["Coordinates"], snap)
    
    center = to_physical(tree["SubhaloPos"][snap_id],snap)
    #particle distances
    r = np.linalg.norm(coords-center,axis=1)
    
    # print(r)
    
    #sort mass and distance based on distance
    sort_idx = np.argsort(r)
    sorted_distances = r[sort_idx]
    
    r_bins = np.logspace(np.log10(0.01), np.log10(r.max()), N_bins+1)
    r_centers = 0.5 * (r_bins[1:] + r_bins[:-1])
    
    m_enc = np.zeros_like(r_centers)

    for i, rmax in enumerate(r_bins[1:]):
        m_enc[i] = p_mass * np.sum(r <= rmax)
    
    v_circ = np.sqrt(G * m_enc / r_centers)
    
    #spline version of rot curve to match GVB paper (4th order spline interpolation)
    
    spline = UnivariateSpline(r_centers, v_circ, k=4, s=0)
    r_fine = np.logspace(np.log10(r_centers.min()), np.log10(r_centers.max()), 1000)
    vc_fine = spline(r_fine)
    
    return r_centers, v_circ, r_fine, vc_fine
        
# calculate manual rmax and vmax using particle data instead of catalog data
def man_struct(tree, snap, particle_data):
    """
    tree: the subhalo tree for which to calculate the concentration
    snap: snapshot to calculate concentration at
    particle_data: output of the groupcat loadSubhalo function (dark matter only)
    """
    _,_,r,v = rot_curve(tree,snap,particle_data)
    max_ind = np.argmax(v)
    return r[max_ind],v[max_ind]

In [ ]:
def get_density_profile(tree,snap,particle_data,NFW=False,manual=False,cut_trunc=False,cut_inner=False):
    """
    tree: the subhalo tree for which to calculate the concentration
    snap: snapshot to calculate concentration at
    particle_data: output of the groupcat loadSubhalo function (dark matter only)
    NFW: if True, also return the theoretical NFW profile for the subhalo
    cut_trunc: cut out data outside the truncation radius (1e-4 of the innermost density)
    cut_inner: cuts data within 0.3 kpc of the center (strictly unresolved by force softening)
    """
    snap_id = snap_to_id(snap,tree)
    
    coords = to_physical(particle_data["Coordinates"], snap)
    
    center = to_physical(tree["SubhaloPos"][snap_id],snap)
    #particle distances
    r = np.linalg.norm(coords-center,axis=1)
    
    # choose radial range
    r_min = 0.1
    r_max = r.max()

    # logarithmic radial bins
    nbins = 30
    bins = np.logspace(np.log10(r_min), np.log10(r_max), nbins + 1)

    # histogram particle counts
    counts, _ = np.histogram(r, bins=bins)

    # shell volumes
    volumes = (4.0/3.0) * np.pi * (bins[1:]**3 - bins[:-1]**3)

    # mass per shell
    mass = counts * p_mass

    # density
    rho = mass / volumes
    
    #density uncertainty
    sigma_N = np.sqrt(counts)
    sigma_rho = sigma_N*p_mass / volumes

    # representative radius (geometric mean is standard)
    r_mid = np.sqrt(bins[1:] * bins[:-1])
    
    if cut_trunc:
        mask = (rho > rho[0]*1e-4) #& (r > 0.3)
        r_mid = r_mid[mask]
        rho = rho[mask]
        sigma_rho = sigma_rho[mask]
        
    if cut_inner:
        mask = r_mid>0.3
        r_mid = r_mid[mask]
        rho = rho[mask]
        sigma_rho = sigma_rho[mask]
    
#     if NFW:
#         if manual is False:
#             vmax = tree['SubhaloVmax'][snap_id]
#             rmax = to_physical(tree['SubhaloVmaxRad'][snap_id],snap)
#         else:
#             rmax,vmax = man_struct(tree,snap,particle_data)
#         #convert rmax to scale radius
#         rs = rmax/2.16
#         rho_s = vmax**2/(4*np.pi*G*(rs**2)*0.216)
#         NFW_rho = []
#         for rad in r_mid:
#             p = rho_s/((rad/rs)*((1+rad/rs)**2))
#             NFW_rho.append(p)
        
#         return r_mid, rho, NFW_rho
    
    return r_mid, rho, sigma_rho

In [ ]:
def get_gnfw_constant(gamma):
    # This is the dimensionless radius where Vmax occurs
    # For NFW it's ~2.16, but for gNFW it shifts.
    # A common approximation for the peak location is x_peak = 2 - gamma 
    # (Note: this is an approximation, but good for a p0 guess)
    x_peak = 2.163 if gamma == 1 else (2.0 - gamma) 
    
    # Numerical integral for M(<r)
    integrand = lambda u: (u**(2-gamma)) * ((1+u)**(gamma-3))
    val, _ = quad(integrand, 0, x_peak)
    
    # Return the equivalent of your '0.216'
    return val / x_peak

In [ ]:
# gnfw function to be optimized
# Function expects x to be log10(r)
def gnfw_log_model(log_r, log_rhos, log_rs, gamma):
    """
    Everything is handled in log10 space to keep gradients 
    stable for the Scipy optimizer.
    log_r: log scaled radii
    log_rhos: log scaled scale density
    log_rs: log scaled scale radius
    gamma: current gamma value of gnfw
    """
    x = (10**log_r) / (10**log_rs)
    
    # log10(rho) = log10(rhos) - gamma*log10(x) - (3-gamma)*log10(1+x)
    term1 = gamma * np.log10(x)
    term2 = (3 - gamma) * np.log10(1 + x)
    
    return log_rhos - term1 - term2

# performs the gnfw fit (using log scaling to weight inner regions)
def gnfw_fit_log(tree, snap):
    """
    tree: the subhalo tree for which to calculate the concentration
    snap: snapshot to calculate concentration at
    """
    snap_id = snap_to_id(snap, tree)
    particle_data = il.snapshot.loadSubhalo(basePath, snap, tree['SubfindID'][snap_id], 1)
    
    r_mid, rho, sigma_rho = get_density_profile(tree, snap, particle_data,cut_trunc=True,cut_inner=True)
    
    mask = (rho > 0) & (sigma_rho > 0)
    r_mid = r_mid[mask]
    rho = rho[mask]
    sigma_rho = sigma_rho[mask]
    
    if len(r_mid) < 5:
        return [np.nan, np.nan, np.nan] # Not enough valid bins
        
    log_r = np.log10(r_mid)
    log_rho = np.log10(rho)
    
    # Logarithmic error propagation
    sigma_log_rho = sigma_rho / (rho * np.log(10))
    
    rm, vm = man_struct(tree, snap, particle_data)
    
    initial_gamma = 1.0
    initial_rs = rm / (2.0 - initial_gamma)
    current_const = get_gnfw_constant(initial_gamma)
    rho_s_guess = vm**2 / (4 * np.pi * G * (initial_rs**2) * current_const)
    
    # 3. ALGORITHMIC FIX: All parameters are roughly the same order of magnitude (0 to 15)
    p0 = [np.log10(rho_s_guess), np.log10(initial_rs), initial_gamma]
    # print(p0)
    bounds = ([-2.0, -2.0, 0.0], [15.0, 3.0, 2.5])
    
    try:
        popt, _ = curve_fit(
            gnfw_log_model, 
            log_r,
            log_rho, 
            p0=p0,
            sigma=sigma_log_rho, 
            absolute_sigma=True, # We want to use the actual log-errors
            bounds=bounds,
            maxfev=10000
        )
        return popt
    except RuntimeError as e:
        # <-- 2. Catch the exact error you got
        print(f"Fit failed to converge for tree: {e}")
        return [np.nan, np.nan, np.nan]
        
    except ValueError as e:
        # Catch other common errors (like NaNs in the data)
        print(f"Value Error for tree: {e}")
        return [np.nan, np.nan, np.nan]
        
    except Exception as e:
        # Catch anything else unexpected
        print(f"Unexpected error for tree: {e}")
        return [np.nan, np.nan, np.nan]

In [ ]:
#model from Green and van den Bosch 2019
def X(fb,cs,param):
    """
    fb: the bound mass fraction
    cs:  the concentration at infall
    mu and nu are fit parameters and functions of fb and cs
    param:  either 'v' or 'r' to indicate vmax or rmax respectively
    """
    mu,nu = gb_mu_nu(fb,cs,param)
    return (2**mu)*(fb**nu)/((1+fb)**mu)

#function to retrive mu and nu values from Green and van den Bosch 2019
def gb_mu_nu(fb,cs,param):
    if param == 'v':
        p0 = 2.980
        p1 = 0.310
        p2 = -0.223
        p3 = -3.308
        p4 = -0.079
        q0 = 0.176
        q1 = -0.008
        q2 = 0.452
    elif param == 'r':
        p0 = 1.021
        p1 = 1.463
        p2 = 0.099
        p3 = -4.643
        p4 = -0.250
        q0 = -0.525
        q1 = -0.065
        q2 = 0.083
    
    mu = p0 + p1*(cs**p2)*np.log10(fb)+p3*(cs**p4)
    nu = q0 + q1*(cs**q2)*np.log10(fb)
    return mu,nu

In [ ]:
# --- Set Up CSV Storage for infall parameters ---
csv_file = 'tidal_features_multihost.csv'
if os.path.exists(csv_file):
    os.remove(csv_file)

# chunk size for file saving during run
chunksize = 5000
rows_chunk = []

print("Extracting physical parameters")

# Master Loop Over Clean Data
for host_id, data in tqdm(final_analysis_store.items(), desc="Processing Hosts"):
    cent_tree = data['central_tree']
    sat_trees = data['satellite_trees']
    infall = data['infall_metrics']
    
    # build position and velocity maps for host from central tree
    host_pos_map = {}
    host_vel_map = {}
    for snap, pos, vel in zip(cent_tree['SnapNum'], cent_tree['SubhaloPos'], cent_tree['SubhaloVel']):
        host_pos_map[snap] = to_physical(pos, snap)
        host_vel_map[snap] = vel

    # Loop Over Subhalos
    for idx, sub_tree in enumerate(sat_trees):
        subhalo_id_z0 = sub_tree['SubhaloID'][0] 
        
        #infall snapshot
        inf_snap = infall['snap'][idx]
        #infall mass
        m0 = infall['mass'][idx]
        #infall rmax
        r0 = infall['rmax'][idx]
        #infall vmax
        v0 = infall['vmax'][idx]
        #infall velocity dispersion
        vdisp0 = infall['vdisp'][idx]
        
        #infall position (physical units)
        sub_pos_inf = to_physical(infall['pos'][idx], inf_snap)
        #infall velocity
        sub_vel_inf = infall['vel'][idx]
        #infall distance
        d0 = np.linalg.norm(sub_pos_inf - host_pos_map[inf_snap])
        #infall radial and tangential velocity
        vr0, vt0 = get_vr_vt(sub_pos_inf, sub_vel_inf, host_pos_map[inf_snap], host_vel_map[inf_snap])
        
        #infall concentration
        c0 = get_concentration(sub_tree, inf_snap)
        #infall velocity anisotropy
        beta0 = get_anisotropy(sub_tree, inf_snap)
        
        #infall axis ratios
        shape0 = compute_shape(sub_tree, inf_snap)
        #infall sphericity
        spher0 = shape0['sphericity']
        #infall triaxiality
        tri0 = shape0['triaxiality']
        
        #gNFW fit
        fit = gnfw_fit_log(sub_tree,inf_snap)
        #infall gNFW gamma
        gamma = fit[2]
        
        #infall time
        t0 = snap_to_time[inf_snap]
        
        # --- Loop Over Snapshots (z=0 back to Infall) ---
        for snap_idx, snap in enumerate(sub_tree['SnapNum']):
            if snap < inf_snap:
                break
                
            # retained mass
            mr = sub_tree['SubhaloMass'][snap_idx]*1e10/h / m0
            # current rmax (physical units)
            r_phys = to_physical(sub_tree['SubhaloVmaxRad'][snap_idx], snap)
            # current normalized rmax
            r_norm = r_phys / r0
            # current normalized vmax
            v_norm = sub_tree['SubhaloVmax'][snap_idx] / v0
            
            # Green and van den Bosch rmax and vmax 
            # for the same retained mass and infall concentration as current subhalo
            gvb_th_vmax = X(mr, c0, 'v')
            gvb_th_rmax = X(mr, c0, 'r')
            
            # current position and velocity
            sub_pos_inst = to_physical(sub_tree['SubhaloPos'][snap_idx], snap)
            sub_vel_inst = sub_tree['SubhaloVel'][snap_idx]
            
            # current distance
            d_inst = np.linalg.norm(sub_pos_inst - host_pos_map[snap])
            # current radial and tangential velocities
            vr, vt = get_vr_vt(sub_pos_inst, sub_vel_inst, host_pos_map[snap], host_vel_map[snap])
            
            # compile row for dataframe
            row = {
                'host_id': host_id,
                'subhalo_id': subhalo_id_z0,
                'snapshot': snap,
                
                'mass_inf': m0,
                'rmax_inf': r0,
                'vmax_inf': v0,
                'vdisp_inf': vdisp0,
                'vtan_inf': vt0,
                'vrad_inf': vr0,
                'concentration_inf': c0,
                'anisotropy_inf': beta0,
                'sphericity_inf': spher0,
                'triaxiality_inf': tri0,
                'gamma_inf':gamma,
                'time_inf': t0,
                'distance_inf': d0,

                'mass': sub_tree['SubhaloMass'][snap_idx] * 1e10 / h,
                'm_retained': mr,
                'rmax_norm': r_norm,
                'vmax_norm': v_norm,
                'vdisp': sub_tree['SubhaloVelDisp'][snap_idx],
                'vrad': vr,
                'vtan': vt,
                'time': snap_to_time[snap],
                'distance': d_inst,
                
                'rmax_gvb': gvb_th_rmax,
                'vmax_gvb': gvb_th_vmax,
                'residual_r': r_norm - gvb_th_rmax,
                'residual_v': v_norm - gvb_th_vmax
            }
            rows_chunk.append(row)

            # --- Chunk Saving Mechanism ---
            if len(rows_chunk) >= chunksize:
                df_chunk = pd.DataFrame(rows_chunk)
                df_chunk.to_csv(csv_file, mode='a', header=not os.path.exists(csv_file), index=False)
                rows_chunk = []

# Flush any remaining rows
if rows_chunk:
    df_chunk = pd.DataFrame(rows_chunk)
    df_chunk.to_csv(csv_file, mode='a', header=not os.path.exists(csv_file), index=False)

print(f"Data extraction complete. Master ML dataset saved to {csv_file}.")

### Additional parameter of subhalo that was tested but not used for the final report

In [ ]:
# 2 body relaxation parameter inspired by van den Bosch et. al 2018 on artificial disruption
# not included with the analysis script because it requires downloading the particle data
def get_tbr(tree,snap):
    snap_id = snap_to_id(snap,tree)
    particle_data = il.snapshot.loadSubhalo(basePath,snap,tree['SubfindID'][snap_id],1)
    coords = to_physical(particle_data["Coordinates"], snap)
    
    center = to_physical(tree["SubhaloPos"][snap_id],snap)
    #particle distances
    r = np.linalg.norm(coords-center,axis=1)
    
    #get rmax in physical units
    rmax = to_physical(tree["SubhaloVmaxRad"][snap_id],snap)
    
    # particles within rmax
    mask = r <= rmax
    N = np.count_nonzero(mask)
    
    # guard against poorly resolved halos
    if N < 10:
        return np.inf
    
    # enclosed mass
    M_enc = N * p_mass  # Msun

    # dynamical time at rmax
    t_dyn = np.sqrt(rmax**3 / (G * M_enc))  # kpc/(km/s)

    # convert to Gyr
    t_dyn *= 0.9778  # (kpc / km/s) → Gyr

    # relaxation time
    t_relax = N /(8*np.log(N)) * t_dyn
    
    return t_relax

def min_t_relax(tree):
    min_tr = np.inf
    min_snap = -1
    
    for i,snap in enumerate(tree["SnapNum"]):
        t_r = get_tbr(tree,snap)
        if t_r < min_tr:
            min_tr = t_r
            min_snap = snap
    
    return min_tr, min_snap